# 11 Deep Learning — Exercises

Practice PyTorch binary classification with the Legionnaires' disease data from Songbai Nursing Home.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# -- CJK font setup (prevents Chinese labels from rendering as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["severe_outcome"] = ((df["hospitalized"] == 1) | (df["outcome"] == "dead")).astype(int)

num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = [
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]

X_df = pd.get_dummies(df[num_cols + cat_cols + bin_cols], drop_first=True)
X_np = X_df.values.astype(np.float32)
scaler = StandardScaler()
X_np[:, 0] = scaler.fit_transform(X_np[:, 0:1]).ravel()

idx = np.arange(len(X_np))
np.random.shuffle(idx)
split = int(0.7 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]

## Question 1: Change the architecture

1. Change the hidden layers from `32 → 16` to `64 → 32 → 16` (three hidden layers)
2. Compute the new model's parameter count
3. Train it with the same early stopping and compare Val AUC
4. Does the more complex architecture perform better? What is the parameter/sample ratio?

In [ ]:
# TODO: Build a model with 3 hidden layers
# TODO: Compute the parameter count
# TODO: Train + early stopping
# TODO: Evaluate Val AUC

## Question 2: Task B — Predict severe cases

1. Change the target variable to `severe_outcome`
2. Train with the `input → 32 → 16 → 1` architecture
3. Plot the learning curve (train/val loss)
4. Compute Val AUC and compare with Task A

In [ ]:
# TODO: y = severe_outcome
# TODO: Build the model, train, early stopping
# TODO: Learning curve
# TODO: Val AUC

## Question 3 (challenge): Add Dropout regularization

1. Add `nn.Dropout(0.3)` after each ReLU
2. Train + early stopping
3. Compare with Dropout vs without Dropout:
   - Train AUC vs Val AUC gap
   - Shape of the learning curve
4. Does Dropout effectively reduce overfitting?

In [ ]:
# TODO: Build a model with Dropout
# TODO: Train + early stopping
# TODO: Compare the Train-Val AUC gap
# TODO: Interpret

## Question 4: A small tabular-data MLP for COVID-19 (COVID-19 scenario)

Build a small neural network in PyTorch to predict severe COVID-19 outcomes.

1. Standardize the features and split into train/validation sets
2. Build a small MLP (`nn.Sequential`), train with `BCEWithLogitsLoss` + `Adam` for ~150 epochs
3. Compute train / validation ROC-AUC
4. Interpret: on small tabular data, does DL have a clear advantage over logistic regression?

In [ ]:
# COVID-19: small tabular-data MLP for binary classification (severe cases)
rng = np.random.default_rng(1104)
n = 900
age = rng.integers(20, 90, n); male = rng.integers(0, 2, n)
diabetes = rng.integers(0, 2, n); vaccinated = rng.binomial(1, 0.6, n)
logit = -6 + 0.06*age + 0.4*male + 0.7*diabetes - 1.2*vaccinated
severe = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, male, diabetes, vaccinated].astype(float); y = severe.astype(float)
print(f"COVID: n={n}, severe rate={y.mean():.1%}, features={X.shape[1]}")

# TODO: Standardize X with StandardScaler, train_test_split (stratify=y, test_size=0.3)
# TODO: Build a small MLP with torch: nn.Sequential(Linear->ReLU->Linear->ReLU->Linear(…,1))
# TODO: Train ~150 epochs with BCEWithLogitsLoss + Adam (full-batch is fine)
# TODO: Get probabilities with sigmoid, compute train / validation ROC-AUC
# TODO: Interpret: on this small tabular dataset, does DL have a clear advantage over logistic regression?

## Question 5: A dengue severity MLP (dengue scenario)

Use a small MLP to predict severe dengue (DHF).

1. Standardize, split, then train a small MLP with 1 hidden layer
2. Compute validation ROC-AUC and compare with the random forest from Chapter 10

In [ ]:
# Dengue: small MLP for severe DHF
rng = np.random.default_rng(1105)
n = 800
age = rng.integers(1, 80, n); secondary = rng.binomial(1, 0.45, n)
platelet = rng.normal(180, 60, n).clip(20, 400); days = rng.integers(1, 8, n)
logit = -2.5 + 1.6*secondary - 0.012*platelet + 0.15*days
dhf = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, secondary, platelet, days].astype(float); y = dhf.astype(float)
print(f"Dengue: n={n}, DHF rate={y.mean():.1%}")

# TODO: Standardize, split, then build and train a small MLP (1 hidden layer of 16 neurons is enough)
# TODO: Compute validation ROC-AUC and compare with your random forest result from Chapter 10

## Question 6: A small NN for influenza hospitalization (influenza scenario)

Use a small neural network to predict influenza hospitalization.

1. Standardize, split, and train a small MLP
2. Report validation ROC-AUC

In [ ]:
# Influenza: small NN for hospitalization prediction
rng = np.random.default_rng(1106)
n = 850
age = rng.integers(0, 95, n); chronic = rng.binomial(1, 0.25, n)
vacc = rng.binomial(1, 0.5, n); onset = rng.integers(0, 6, n)
logit = -3.5 + 0.05*age + 1.0*chronic - 0.8*vacc + 0.25*onset
hosp = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, chronic, vacc, onset].astype(float); y = hosp.astype(float)
print(f"Flu: n={n}, hospitalization rate={y.mean():.1%}")

# TODO: Standardize, split, then train a small MLP and report validation ROC-AUC

## Question 7: A small NN for tuberculosis prognosis (TB scenario)

Use a small neural network to predict TB treatment success.

1. Standardize, split, and train a small MLP
2. Report validation ROC-AUC, and note the direction of adherence's effect

In [ ]:
# Tuberculosis: small NN for treatment outcome prognosis
rng = np.random.default_rng(1107)
n = 800
age = rng.integers(18, 85, n); mdr = rng.binomial(1, 0.15, n)
hiv = rng.binomial(1, 0.1, n); adher = rng.uniform(0.4, 1.0, n)
logit = 2.0 - 1.8*mdr - 1.2*hiv + 3.0*(adher-0.7)
success = rng.binomial(1, 1/(1+np.exp(-logit)))
X = np.c_[age, mdr, hiv, adher].astype(float); y = success.astype(float)
print(f"TB: n={n}, treatment success rate={y.mean():.1%}")

# TODO: Standardize, split, then train a small MLP and report validation ROC-AUC
# TODO: Pay special attention to the direction of adherence's effect on success

## Question 8 (challenge): Demonstrating small-sample overfitting (model diagnostics scenario)

Deliberately demonstrate overfitting with "a very small sample + many noise features + an oversized network."

1. Standardize and split (test_size=0.4)
2. Train an oversized network (2 layers of 128 neurons each) for more epochs
3. Record and plot the train loss and validation loss curves
4. Observe train loss decreasing while val loss turns around and rises; find the epoch where you should have stopped early
5. Interpret: why do small samples + large networks overfit easily? What should you do with small epidemiological datasets?

In [ ]:
# Small-sample overfitting demo: very small n, many noise features (challenge)
rng = np.random.default_rng(1108)
n = 90                      # Very small sample
signal = rng.normal(0, 1, n)
noise = rng.normal(0, 1, (n, 30))   # 30 pure noise features
y = (1/(1+np.exp(-(1.5*signal))) > rng.uniform(0, 1, n)).astype(float)
X = np.c_[signal, noise].astype(float)   # 1 signal feature + 30 noise features
print(f"n={n}, features={X.shape[1]} (only 1 carries signal), positive rate={y.mean():.1%}")

# TODO: Standardize, train_test_split (test_size=0.4)
# TODO: Train an "oversized" network (e.g. 2 layers of 128 neurons each) for more epochs
# TODO: Record train loss and validation loss for each epoch, plot as two curves
# TODO: Observe train loss keeps decreasing while val loss turns around and rises → overfitting
# TODO: Interpret: why do small samples + large networks overfit easily? What would you do with small epidemiological data?